# Risk Calculation and Analysis Data Preparation

This notebook builds the final cell-month risk dataset for the California coast (2020–2025) by combining WhaleWatch whale presence information with GFW vessel activity data. The resulting outputs are intended to serve as the input data for the next stages of analysis, including spatial hotspot analysis (LISA) and temporal risk pattern analysis.

In [3]:
# 0. imports, paths, constants
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import box

# Project paths
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "Data"
PROCESSED_DIR = DATA_DIR / "processed"

# Input files
WHALE_FILE = PROCESSED_DIR / "whale_watch_2020_2025_combined.csv"
GFW_FILE = DATA_DIR / "gfw_presence_2020_2025_all.csv"
STATES_SHP = DATA_DIR / "tl_2024_us_state" / "tl_2024_us_state.shp"

# Output files
BUFFER_FILE = PROCESSED_DIR / "ca_200km_buffer.gpkg"
FISHNET_FILE = PROCESSED_DIR / "fishnet_025_buffer.gpkg"
RISK_MASTER_CSV = PROCESSED_DIR / "whale_vessel_risk_master.csv"
RISK_MASTER_GPKG = PROCESSED_DIR / "whale_vessel_risk_master.gpkg" 

# Constants
EXPECTED_VESSELS = {"cargo", "fishing"}
SPEED_WEIGHTS = {
    "<2": 0.2,
    "2-4": 0.2,
    "4-6": 0.2,
    "6-10": 0.2,
    "10-15": 0.6,
    "15-25": 1.0,
    ">25": 1.2,
}
EXPECTED_SPEED_BANDS = set(SPEED_WEIGHTS.keys())

CELL_SIZE = 0.25
BUFFER_KM = 200

print("Whale file exists:", WHALE_FILE.exists(), WHALE_FILE)
print("GFW file exists:", GFW_FILE.exists(), GFW_FILE)
print("State shapefile exists:", STATES_SHP.exists(), STATES_SHP)

Whale file exists: True Data/processed/whale_watch_2020_2025_combined.csv
GFW file exists: True Data/gfw_presence_2020_2025_all.csv
State shapefile exists: True Data/tl_2024_us_state/tl_2024_us_state.shp


## 1. Data Preparation
### 1-1. Load WhaleWatch (minimal columns only)

In [4]:
whale_cols = [
    "longitude", "latitude", "year", "month",
    "percent", "fitmean", "density", "has_model_output"
]

ww = pd.read_csv(WHALE_FILE, usecols=whale_cols)

print(f"WhaleWatch longitude range: {ww['longitude'].min():.2f} to {ww['longitude'].max():.2f}")
print(f"WhaleWatch latitude range: {ww['latitude'].min():.2f} to {ww['latitude'].max():.2f}")

# Keep only rows where model output exists
ww_valid = ww[ww["has_model_output"]].copy()

print("WhaleWatch total rows:", len(ww))
print("WhaleWatch valid rows:", len(ww_valid))
print("WhaleWatch missing model-output rows:", (~ww["has_model_output"]).sum())

print("WhaleWatch unique years:", sorted(ww_valid["year"].unique()))
print("WhaleWatch unique months:", sorted(ww_valid["month"].unique()))

WhaleWatch longitude range: -135.00 to -115.00
WhaleWatch latitude range: 30.00 to 49.00
WhaleWatch total rows: 399168
WhaleWatch valid rows: 240785
WhaleWatch missing model-output rows: 158383
WhaleWatch unique years: [2020, 2021, 2022, 2023, 2024, 2025]
WhaleWatch unique months: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


### 1-2. Load GFW (minimal columns only)

In [5]:
gfw_cols = [
    "lon", "lat", "year", "month",
    "hours", "vessel_type", "speed_band"
]
gfw = pd.read_csv(GFW_FILE, usecols=gfw_cols)

# Drop rows with missing core fields
gfw = gfw.dropna(subset=["lon", "lat", "year", "month", "hours", "vessel_type", "speed_band"]).copy()

# Keep only vessel types used in project
gfw = gfw[gfw["vessel_type"].isin(EXPECTED_VESSELS)].copy()

# Keep only expected speed bands
gfw = gfw[gfw["speed_band"].isin(EXPECTED_SPEED_BANDS)].copy()

# Memory optimization
gfw["vessel_type"] = gfw["vessel_type"].astype("category")
gfw["speed_band"] = gfw["speed_band"].astype("category")

print("GFW rows after cleaning:", len(gfw))
print("GFW unique years:", sorted(gfw["year"].unique()))
print("GFW unique months:", sorted(gfw["month"].unique()))
print("GFW vessel types:", sorted(gfw["vessel_type"].astype(str).unique()))
print("GFW speed bands:", sorted(gfw["speed_band"].astype(str).unique()))
print("GFW longitude range:", gfw["lon"].min(), "to", gfw["lon"].max())
print("GFW latitude range:", gfw["lat"].min(), "to", gfw["lat"].max())

GFW rows after cleaning: 4370448
GFW unique years: [2020, 2021, 2022, 2023, 2024, 2025]
GFW unique months: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
GFW vessel types: ['cargo', 'fishing']
GFW speed bands: ['10-15', '15-25', '2-4', '4-6', '6-10', '<2', '>25']
GFW longitude range: -135.0 to -115.1
GFW latitude range: 30.0 to 49.0


### 1-3. Keep only year-month combinations shared by both datasets

In [6]:
ww_months = ww_valid[["year", "month"]].drop_duplicates()
gfw_months = gfw[["year", "month"]].drop_duplicates()

common_months = ww_months.merge(gfw_months, on=["year", "month"], how="inner")

ww_valid = ww_valid.merge(common_months, on=["year", "month"], how="inner")
gfw = gfw.merge(common_months, on=["year", "month"], how="inner")

print("Shared (year, month) combinations:", len(common_months))
print("WhaleWatch rows after month filter:", len(ww_valid))
print("GFW rows after month filter:", len(gfw))

Shared (year, month) combinations: 64
WhaleWatch rows after month filter: 240785
GFW rows after month filter: 3894214


### 1-4. Check state shapefile is readable

In [7]:
states = gpd.read_file(STATES_SHP)

print("States CRS:", states.crs)
print("State columns:", states.columns.tolist())

ca = states[states["NAME"] == "California"].copy()
print("California rows found:", len(ca))

if len(ca) != 1:
    raise ValueError("California polygon was not found correctly in the state shapefile.")

States CRS: EPSG:4269
State columns: ['REGION', 'DIVISION', 'STATEFP', 'STATENS', 'GEOID', 'GEOIDFQ', 'STUSPS', 'NAME', 'LSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry']
California rows found: 1


### 1-5. Reproject California polygon for distance-based work and joining

In [8]:
print("Original California CRS:", ca.crs)

# For accurate distance/buffer calculation
ca_3310 = ca.to_crs(epsg=3310)
print("California projected CRS:", ca_3310.crs)
print("California bounds in EPSG:3310:")
print(ca_3310.total_bounds)

Original California CRS: EPSG:4269
California projected CRS: EPSG:3310
California bounds in EPSG:3310:
[-380101.0656607  -605252.61217882  540037.54685041  450446.71159829]


## 2. Spatial Framework (Grid)
### 2-0. Create 200 km buffer from California boundary

In [9]:
# BUFFER_KM was already defined above
buffer_m = BUFFER_KM * 1000 

# Buffer in projected CRS for accurate distance
ca_buffer_3310 = ca_3310.copy()
ca_buffer_3310["geometry"] = ca_buffer_3310.geometry.buffer(buffer_m)

print("Buffered California CRS:", ca_buffer_3310.crs)
print("Buffered California bounds in EPSG:3310:")
print(ca_buffer_3310.total_bounds)

# Reproject buffer back to WGS84 for later fishnet overlay / joins
ca_buffer_wgs84 = ca_buffer_3310.to_crs(epsg=4326)

print("Buffered California CRS in WGS84:", ca_buffer_wgs84.crs)
print("Buffered California bounds in EPSG:4326:")
print(ca_buffer_wgs84.total_bounds)

# Save buffer
ca_buffer_wgs84.to_file(BUFFER_FILE, driver="GPKG")
print("Saved buffer to:", BUFFER_FILE)

Buffered California CRS: EPSG:3310
Buffered California bounds in EPSG:3310:
[-579918.1334305  -805252.56169665  739911.38228698  650243.39491612]
Buffered California CRS in WGS84: EPSG:4326
Buffered California bounds in EPSG:4326:
[-126.84121804   30.72104585 -111.95916027   43.81602217]
Saved buffer to: Data/processed/ca_200km_buffer.gpkg


### 2-1. Build a 0.25° fishnet aligned to the WhaleWatch lattice

In [10]:
def cell_polygon(lon, lat, size=CELL_SIZE):
    """Create a square polygon centered on (lon, lat)."""
    half = size / 2
    return box(lon - half, lat - half, lon + half, lat + half)

# Use unique WhaleWatch grid centers
centers = (
    ww_valid[["longitude", "latitude"]]
    .drop_duplicates()
    .rename(columns={"longitude": "cell_lon", "latitude": "cell_lat"})
    .copy()
)

# Create square polygons centered on each WhaleWatch grid point
fishnet = gpd.GeoDataFrame(
    centers,
    geometry=[
        cell_polygon(lon, lat)
        for lon, lat in zip(centers["cell_lon"], centers["cell_lat"])
    ],
    crs="EPSG:4326"
)

print("Fishnet cells before buffer filter:", len(fishnet))
print("Fishnet CRS:", fishnet.crs)
fishnet.head()

Fishnet cells before buffer filter: 3781
Fishnet CRS: EPSG:4326


,cell_lon,cell_lat,geometry
0,-134.75,30.75,"POLYGON ((-134.625 30.625, -134.625 30.875, -1..."
1,-134.75,31.00,"POLYGON ((-134.625 30.875, -134.625 31.125, -1..."
2,-134.75,31.25,"POLYGON ((-134.625 31.125, -134.625 31.375, -1..."
3,-134.75,31.50,"POLYGON ((-134.625 31.375, -134.625 31.625, -1..."
4,-134.75,31.75,"POLYGON ((-134.625 31.625, -134.625 31.875, -1..."


### 2-2. Clip the fishnet to the 200 km coastal buffer

In [11]:
# Merge buffer geometry into a single shape
study_geom = ca_buffer_wgs84.geometry.union_all()

# Keep only fishnet cells that intersect the study area
fishnet_in_buffer = fishnet[fishnet.geometry.intersects(study_geom)].copy()

# Sort and assign cell_id
fishnet_in_buffer = fishnet_in_buffer.sort_values(["cell_lat", "cell_lon"]).reset_index(drop=True)
fishnet_in_buffer["cell_id"] = fishnet_in_buffer.index.astype(int)

# Reorder columns
fishnet_in_buffer = fishnet_in_buffer[["cell_id", "cell_lon", "cell_lat", "geometry"]]

# Save
fishnet_in_buffer.to_file(FISHNET_FILE, driver="GPKG")

print("Fishnet cells inside buffer:", len(fishnet_in_buffer))
print("Fishnet CRS:", fishnet_in_buffer.crs)
fishnet_in_buffer.head()

Fishnet cells inside buffer: 676
Fishnet CRS: EPSG:4326


,cell_id,cell_lon,cell_lat,geometry
0,0,-118.00,30.75,"POLYGON ((-117.875 30.625, -117.875 30.875, -1..."
1,1,-117.75,30.75,"POLYGON ((-117.625 30.625, -117.625 30.875, -1..."
2,2,-117.50,30.75,"POLYGON ((-117.375 30.625, -117.375 30.875, -1..."
3,3,-117.25,30.75,"POLYGON ((-117.125 30.625, -117.125 30.875, -1..."
4,4,-117.00,30.75,"POLYGON ((-116.875 30.625, -116.875 30.875, -1..."


## 3. Spatial Aggregation & Join
### 3-1. Aggregate WhaleWatch to one record per (cell, year, month)

In [12]:
# Step 1: match WhaleWatch points to fishnet cells by coordinate
ww_cells = ww_valid.rename(columns={"longitude": "cell_lon", "latitude": "cell_lat"}).copy()

# Keep only WhaleWatch rows that fall inside the buffered fishnet
ww_cells = ww_cells.merge(
    fishnet_in_buffer[["cell_id", "cell_lon", "cell_lat"]],
    on=["cell_lon", "cell_lat"],
    how="inner"
)
print("WhaleWatch rows after keeping only buffered cells:", len(ww_cells))


# Step 2: check whether multiple rows exist for the same cell-year-month
dup_check = (
    ww_cells
    .groupby(["cell_id", "cell_lon", "cell_lat", "year", "month"])
    .size()
    .reset_index(name="n_rows")
)

print("Maximum rows per (cell, year, month):", dup_check["n_rows"].max())
print("Groups with more than 1 row:", (dup_check["n_rows"] > 1).sum())

# Step 3: no aggregation needed because each cell-year-month already has one row
ww_agg = ww_cells[[
    "cell_id", "cell_lon", "cell_lat",
    "year", "month",
    "percent", "fitmean", "density"
]].copy()

# sort for stable, readable output
ww_agg = ww_agg.sort_values(
    ["cell_id", "year", "month"]
).reset_index(drop=True)

print("WhaleWatch final rows:", len(ww_agg))
ww_agg.head()

WhaleWatch rows after keeping only buffered cells: 43182
Maximum rows per (cell, year, month): 1
Groups with more than 1 row: 0
WhaleWatch final rows: 43182


,cell_id,cell_lon,cell_lat,year,month,percent,fitmean,density
0,0,-118.0,30.75,2020,1,59.257413,0.592574,0.137342
1,0,-118.0,30.75,2020,2,28.734810,0.287348,0.073861
2,0,-118.0,30.75,2020,3,30.032749,0.300327,0.175277
3,0,-118.0,30.75,2020,4,81.190221,0.811902,0.774399
4,0,-118.0,30.75,2020,5,78.732336,0.787323,0.631794


##  3-2. Snap GFW to WhaleWatch 0.25° grid and aggregate hours

In [13]:
# Use the WhaleWatch lattice origin from the fishnet
lon_origin = fishnet["cell_lon"].min()
lat_origin = fishnet["cell_lat"].min()

print("WhaleWatch grid origin:", lon_origin, lat_origin)
print("Cell size:", CELL_SIZE)

def snap_to_whale_grid(values, origin, step=CELL_SIZE):
    """
    Snap coordinates to the nearest WhaleWatch 0.25° lattice value.
    """
    values = np.asarray(values)
    snapped = origin + np.floor(((values - origin) / step) + 0.5) * step
    return np.round(snapped, 2)

# Copy GFW and create snapped WhaleWatch cell coordinates
gfw_snap = gfw.copy()

gfw_snap["cell_lon"] = snap_to_whale_grid(gfw_snap["lon"], lon_origin)
gfw_snap["cell_lat"] = snap_to_whale_grid(gfw_snap["lat"], lat_origin)

print("GFW rows before snap/filter:", len(gfw_snap))
print("Snapped longitude range:", gfw_snap["cell_lon"].min(), "to", gfw_snap["cell_lon"].max())
print("Snapped latitude range:", gfw_snap["cell_lat"].min(), "to", gfw_snap["cell_lat"].max())


# Keep only snapped cells that exist inside the buffered fishnet
gfw_snap = gfw_snap.merge(
    fishnet_in_buffer[["cell_id", "cell_lon", "cell_lat"]],
    on=["cell_lon", "cell_lat"],
    how="inner"
)

print("GFW rows after keeping only buffered snapped cells:", len(gfw_snap))

# Aggregate vessel hours within each WhaleWatch cell-month-vessel_type-speed_band
gfw_agg = (
    gfw_snap
    .groupby(
        ["cell_id", "cell_lon", "cell_lat", "year", "month", "vessel_type", "speed_band"],
        as_index=False,
        observed=True,
        sort=False
    )["hours"]
    .sum()
    .rename(columns={"hours": "hours_in_band"})
)

print("Aggregated GFW rows:", len(gfw_agg))
gfw_agg.head()

WhaleWatch grid origin: -134.75 30.5
Cell size: 0.25
GFW rows before snap/filter: 3894214
Snapped longitude range: -135.0 to -115.0
Snapped latitude range: 30.0 to 49.0
GFW rows after keeping only buffered snapped cells: 973237
Aggregated GFW rows: 173070


,cell_id,cell_lon,cell_lat,year,month,vessel_type,speed_band,hours_in_band
0,387,-122.50,37.00,2020,1,cargo,<2,7
1,312,-121.75,35.50,2020,1,cargo,<2,20
2,472,-124.50,38.75,2020,1,cargo,<2,6
3,483,-124.50,39.00,2020,1,cargo,<2,3
4,506,-124.50,39.50,2020,1,cargo,<2,1


##### Do not join yet: GFW still has multiple rows per cell-month. 
- First compute exposure, then collapse to one row per cell-month, then join to WhaleWatch.

## 4. Vessel Exposure (Speed-Weighted)
### 4-1. Compute speed-weighted vessel exposure

In [14]:
# Map each speed band to its literature-based weight
gfw_exposure = gfw_agg.copy()
gfw_exposure["speed_weight"] = gfw_exposure["speed_band"].map(SPEED_WEIGHTS)

# Safety check: no missing weights
if gfw_exposure["speed_weight"].isna().any():
    missing_bands = sorted(gfw_exposure.loc[gfw_exposure["speed_weight"].isna(), "speed_band"].astype(str).unique())
    raise ValueError(f"Missing speed weights for bands: {missing_bands}")

# Compute the weighted exposure contribution of each vessel_type / speed_band row
gfw_exposure["weighted_hours"] = (
    gfw_exposure["hours_in_band"] * gfw_exposure["speed_weight"]
)

print("Rows in weighted GFW table:", len(gfw_exposure))
print("Total raw hours:", gfw_exposure["hours_in_band"].sum())
print("Total weighted hours:", gfw_exposure["weighted_hours"].sum())

gfw_exposure.head()

Rows in weighted GFW table: 173070
Total raw hours: 3678745
Total weighted hours: 1182378.0000000002


,cell_id,cell_lon,cell_lat,year,month,vessel_type,speed_band,hours_in_band,speed_weight,weighted_hours
0,387,-122.50,37.00,2020,1,cargo,<2,7,0.2,1.4
1,312,-121.75,35.50,2020,1,cargo,<2,20,0.2,4.0
2,472,-124.50,38.75,2020,1,cargo,<2,6,0.2,1.2
3,483,-124.50,39.00,2020,1,cargo,<2,3,0.2,0.6
4,506,-124.50,39.50,2020,1,cargo,<2,1,0.2,0.2


### 4-2. Collapse to one exposure value per (cell, year, month)

In [15]:
exposure_agg = (
    gfw_exposure
    .groupby(["cell_id", "cell_lon", "cell_lat", "year", "month"], as_index=False)
    .agg(
        exposure=("weighted_hours", "sum"),
        raw_hours_total=("hours_in_band", "sum")
    )
    .sort_values(["cell_id", "year", "month"])
    .reset_index(drop=True)
)

print("Exposure rows:", len(exposure_agg))
exposure_agg.head()

Exposure rows: 42531


,cell_id,cell_lon,cell_lat,year,month,exposure,raw_hours_total
0,0,-118.0,30.75,2020,1,2.6,3
1,0,-118.0,30.75,2020,2,3.6,6
2,0,-118.0,30.75,2020,3,0.6,1
3,0,-118.0,30.75,2020,4,3.8,7
4,0,-118.0,30.75,2020,5,2.4,4


##### Tables used at this stage:
- `ww_agg`: WhaleWatch cell-month table
- `exposure_agg`: final vessel exposure cell-month table used for risk join
- `gfw_agg`: GFW summary with `vessel_type` and `speed_band` retained for later analysis

### 4-3. Join WhaleWatch and collapsed vessel exposure on the cell-month unit

In [16]:
# Join on the common cell-month keys
risk_master = ww_agg.merge(
    exposure_agg,
    on=["cell_id", "cell_lon", "cell_lat", "year", "month"],
    how="left",
    validate="one_to_one"
)

# If a whale cell-month has no matched vessel exposure, set exposure to 0
# Exposure NA after the join means no matched vessel exposure, so fill with 0.
risk_master["exposure"] = risk_master["exposure"].fillna(0)
risk_master["raw_hours_total"] = risk_master["raw_hours_total"].fillna(0)

print("Joined risk table rows:", len(risk_master))
print("Missing exposure rows filled with 0:", (risk_master["exposure"] == 0).sum())

risk_master.head()

Joined risk table rows: 43182
Missing exposure rows filled with 0: 732


,cell_id,cell_lon,cell_lat,year,month,percent,fitmean,density,exposure,raw_hours_total
0,0,-118.0,30.75,2020,1,59.257413,0.592574,0.137342,2.6,3.0
1,0,-118.0,30.75,2020,2,28.734810,0.287348,0.073861,3.6,6.0
2,0,-118.0,30.75,2020,3,30.032749,0.300327,0.175277,0.6,1.0
3,0,-118.0,30.75,2020,4,81.190221,0.811902,0.774399,3.8,7.0
4,0,-118.0,30.75,2020,5,78.732336,0.787323,0.631794,2.4,4.0


## 5. Risk Calculation
### 5-1. Compute whale-vessel risk metrics

In [17]:
# Convert WhaleWatch percent (0-100) to probability (0-1)
risk_master["whale_prob"] = risk_master["percent"] / 100.0

# Composite risk
risk_master["risk"] = risk_master["whale_prob"] * risk_master["exposure"]

# Log-dampened variant
risk_master["risk_log"] = risk_master["whale_prob"] * np.log1p(risk_master["exposure"])

print("Risk table rows:", len(risk_master))
print("Risk summary:")
print(risk_master[["whale_prob", "exposure", "risk", "risk_log"]].describe())

risk_master.head()

Risk table rows: 43182
Risk summary:
         whale_prob      exposure          risk      risk_log
count  43182.000000  43182.000000  43182.000000  43182.000000
mean       0.286871     27.337937     10.064419      0.751543
std        0.291565     76.940834     43.737521      0.972286
min        0.000099      0.000000      0.000000      0.000000
25%        0.043978      4.600000      0.328349      0.087386
50%        0.171092     10.000000      1.438187      0.353558
75%        0.467445     22.800000      5.272748      1.029464
max        0.996992   2492.400000   2125.337945      7.056912


,cell_id,cell_lon,cell_lat,year,month,percent,fitmean,density,exposure,raw_hours_total,whale_prob,risk,risk_log
0,0,-118.0,30.75,2020,1,59.257413,0.592574,0.137342,2.6,3.0,0.592574,1.540693,0.759048
1,0,-118.0,30.75,2020,2,28.734810,0.287348,0.073861,3.6,6.0,0.287348,1.034453,0.438509
2,0,-118.0,30.75,2020,3,30.032749,0.300327,0.175277,0.6,1.0,0.300327,0.180196,0.141155
3,0,-118.0,30.75,2020,4,81.190221,0.811902,0.774399,3.8,7.0,0.811902,3.085228,1.273563
4,0,-118.0,30.75,2020,5,78.732336,0.787323,0.631794,2.4,4.0,0.787323,1.889576,0.963507


### 5-2. Attach geometry and save outputs

In [18]:
risk_gdf = risk_master.merge(
    fishnet_in_buffer[["cell_id", "geometry"]],
    on="cell_id",
    how="left",
    validate="many_to_one"
)

risk_gdf = gpd.GeoDataFrame(risk_gdf, geometry="geometry", crs="EPSG:4326")

# Save table outputs
risk_master.to_csv(RISK_MASTER_CSV, index=False)
risk_gdf.to_file(RISK_MASTER_GPKG, driver="GPKG")

print("Saved CSV:", RISK_MASTER_CSV)
print("Saved GPKG:", RISK_MASTER_GPKG)

risk_gdf.head()

Saved CSV: Data/processed/whale_vessel_risk_master.csv
Saved GPKG: Data/processed/whale_vessel_risk_master.gpkg


,cell_id,cell_lon,cell_lat,year,month,percent,fitmean,density,exposure,raw_hours_total,whale_prob,risk,risk_log,geometry
0,0,-118.0,30.75,2020,1,59.257413,0.592574,0.137342,2.6,3.0,0.592574,1.540693,0.759048,"POLYGON ((-117.875 30.625, -117.875 30.875, -1..."
1,0,-118.0,30.75,2020,2,28.734810,0.287348,0.073861,3.6,6.0,0.287348,1.034453,0.438509,"POLYGON ((-117.875 30.625, -117.875 30.875, -1..."
2,0,-118.0,30.75,2020,3,30.032749,0.300327,0.175277,0.6,1.0,0.300327,0.180196,0.141155,"POLYGON ((-117.875 30.625, -117.875 30.875, -1..."
3,0,-118.0,30.75,2020,4,81.190221,0.811902,0.774399,3.8,7.0,0.811902,3.085228,1.273563,"POLYGON ((-117.875 30.625, -117.875 30.875, -1..."
4,0,-118.0,30.75,2020,5,78.732336,0.787323,0.631794,2.4,4.0,0.787323,1.889576,0.963507,"POLYGON ((-117.875 30.625, -117.875 30.875, -1..."


In [19]:
# Check
print(risk_master[["whale_prob", "exposure", "risk", "risk_log"]].isna().sum())
print((risk_master["risk"] < 0).sum())
print((risk_master["risk_log"] < 0).sum())

whale_prob    0
exposure      0
risk          0
risk_log      0
dtype: int64
0
0
